# Preprocessing sebagai Fungsi
**Input: `combine.csv`** (haystack gabungan: scraping kelas + injeksi + injeksi manual) dan **`data_putusan_merek.csv`** (gold cases).

Notebook ini:
1. Membersihkan haystack (buang duplikat id).
2. **Menambal cakupan**: auto-injeksi nama-saja untuk merek gold yang masih belum ada di haystack, agar setiap merek pembanding punya target retrieval.
3. Menerapkan preprocessing **berbeda per dimensi** (base untuk tekstual/semantik; base + fonetik-ID untuk fonetik), sebagai kolom turunan tanpa menimpa nama asli.

Prinsip kunci: normalisasi fonetik **tidak** mencemari dimensi tekstual — itu sebabnya ada dua kolom (`nama_base`, `nama_phon`).


In [1]:
# Instalasi dependensi (jalankan sekali)
!pip install rapidfuzz -q
print('rapidfuzz terpasang.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 81.2 MB/s eta 0:00:00
rapidfuzz terpasang.


## 0. Setup & upload

In [2]:
import re, unicodedata, os
import pandas as pd
from rapidfuzz import fuzz

HAYSTACK_PATH = "combine.csv"
GOLD_PATH = "data_putusan_merek.csv"
AKTIFKAN_FONETIK_ID = True   # set False untuk eksperimen ablasi (perbandingan dgn/tanpa normalisasi fonetik)

if not os.path.exists(HAYSTACK_PATH) or not os.path.exists(GOLD_PATH):
    from google.colab import files
    print("Upload combine.csv dan data_putusan_merek.csv:")
    files.upload()
print("Setup selesai.")

Upload combine.csv dan data_putusan_merek.csv:


Saving combine.csv to combine.csv
Setup selesai.


## 1. Fungsi preprocessing (sumber tunggal — impor ini juga di Langkah C)

In [3]:
LEET = {"0":"o","1":"i","3":"e","4":"a","5":"s","7":"t"}
FIGURATIF = r"(lukisan|logo|gambar|device)"
PHONETIC_RULES = [("sy","sh"), ("ny","n"), ("kh","h")]   # 'c'->'k' sengaja TIDAK dipakai (c ID = /tʃ/)

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(c))

def base_normalize(nama):
    s = strip_accents(nama).lower()
    s = re.sub(r"\+\s*" + FIGURATIF + r"\b", " ", s)
    s = re.sub(r"\b" + FIGURATIF + r"\b", " ", s)
    s = re.sub(r"(?<=[a-z])[013457](?=[a-z])", lambda m: LEET.get(m.group(0), m.group(0)), s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def phonetic_normalize(base_str, aktif=True):
    if not aktif:
        return base_str
    x = base_str
    for a, b in PHONETIC_RULES:
        x = x.replace(a, b)
    x = re.sub(r"([aeiou])\1+", r"\1", x)
    return re.sub(r"\s+", " ", x).strip()

def normcmp(s):
    s = str(s).upper().strip()
    s = re.sub(r"\+\s*(LUKISAN|LOGO|GAMBAR|DEVICE).*$", "", s)
    s = re.sub(r"[.,\-/&]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

print("Fungsi siap.")

Fungsi siap.


## 2. Muat haystack, buang duplikat id

In [4]:
hay = pd.read_csv(HAYSTACK_PATH, dtype=str, encoding="utf-8-sig").fillna("")
hay.columns = [c.strip().lstrip("\ufeff") for c in hay.columns]
n0 = len(hay)
hay = hay.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)
print(f"Haystack: {n0} -> {len(hay)} (buang {n0-len(hay)} duplikat id)")
print("sumber:", hay["sumber"].value_counts().to_dict() if "sumber" in hay.columns else "-")

Haystack: 13512 -> 13508 (buang 4 duplikat id)
sumber: {'scraping_kelas': 13187, 'injeksi': 209, 'injeksi_manual': 112}


## 3. Tambal cakupan: auto-injeksi nama-saja untuk merek gold yang belum ada
Merek gold yang tak ditemukan di PDKI (mis. merek asing/beda ejaan jauh) disuntik dari nama di putusan, ditandai `sumber="injeksi_manual_nama"`, agar retrieval punya target.


In [5]:
from collections import defaultdict
gold_raw = pd.read_csv(GOLD_PATH, dtype=str, encoding="utf-8-sig").fillna("")
gold_raw.columns = [c.strip().lstrip("\ufeff") for c in gold_raw.columns]

RANK = {"PK":3, "PENINJAUAN KEMBALI":3, "KASASI":2, "NIAGA":1}
groups = defaultdict(list)
for _, r in gold_raw.iterrows():
    groups[tuple(sorted([normcmp(r["merek_a"]), normcmp(r["merek_b"])]))].append(r)
final = [sorted(g, key=lambda r: RANK.get(str(r["tingkat"]).strip().upper(), 0), reverse=True)[0]
         for g in groups.values()]
main = [r for r in final if r["label_usulan"] != "reputasi_dominan"]

merek_gold = {}
for r in main:
    for k in ("merek_a", "merek_b"):
        v = str(r[k]).strip()
        if v:
            merek_gold[normcmp(v)] = v

hay_norm = [normcmp(n) for n in hay["nama_merek"]]
hay_set = set(hay_norm)
belum = []
for mn, asli in merek_gold.items():
    if mn in hay_set:
        continue
    if max((fuzz.ratio(mn, x) for x in hay_norm), default=0) >= 90:
        continue
    belum.append(asli)

def pecah(nama):
    return [p.strip() for p in re.split(r"[;/]", nama) if p.strip()]

rows, i = [], 0
for asli in belum:
    for part in pecah(asli):
        rows.append({"id": f"GOLDNAME_{i}", "kelas": "", "nama_merek": part, "pemilik": "",
                     "nomor_permohonan": "", "tanggal_permohonan": "", "status_permohonan": "",
                     "query_asal": asli, "sumber": "injeksi_manual_nama"})
        i += 1
inj_nama = pd.DataFrame(rows)
print(f"Merek gold belum tercakup: {len(belum)} -> {len(inj_nama)} baris auto-injeksi")

for c in hay.columns:
    if c not in inj_nama.columns:
        inj_nama[c] = ""
hay = pd.concat([hay, inj_nama[hay.columns]], ignore_index=True).drop_duplicates(subset=["id"]).reset_index(drop=True)
print("Haystack setelah tambal cakupan:", len(hay))

Merek gold belum tercakup: 39 -> 51 baris auto-injeksi
Haystack setelah tambal cakupan: 13559


## 4. Terapkan preprocessing ke haystack

In [6]:
pp = hay["nama_merek"].apply(lambda x: (base_normalize(x),))
hay["nama_base"] = hay["nama_merek"].apply(base_normalize)
hay["nama_phon"] = hay["nama_base"].apply(lambda b: phonetic_normalize(b, AKTIFKAN_FONETIK_ID))

kosong = (hay["nama_base"] == "").sum()
hay_valid = hay[hay["nama_base"] != ""].reset_index(drop=True)
hay_didaftar = hay_valid[hay_valid["status_permohonan"].str.contains("Didaftar", case=False, na=False)].reset_index(drop=True)
print(f"Valid (ada unsur kata): {len(hay_valid)} (buang {kosong} murni figuratif)")
print(f"Subset Didaftar (basis pembanding retrieval): {len(hay_didaftar)}")
print(f"Seluruh {len(hay_valid)} baris tetap dipakai sbg corpus FastText.")
hay_valid[["nama_merek","nama_base","nama_phon","status_permohonan","sumber"]].head()

Valid (ada unsur kata): 13514 (buang 45 murni figuratif)
Subset Didaftar (basis pembanding retrieval): 7650
Seluruh 13514 baris tetap dipakai sbg corpus FastText.


,nama_merek,nama_base,nama_phon,status_permohonan,sumber
0,REVO CLEAN,revo clean,revo clean,(TM) Dianggap Ditarik Kembali,scraping_kelas
1,Cireng Bawell,cireng bawell,cireng bawell,(TM) Dianggap Ditarik Kembali,scraping_kelas
2,FIT WITH HONEY,fit with honey,fit with honey,(TM) Dianggap Ditarik Kembali,scraping_kelas
3,TeKo,teko,teko,(TM) Dianggap Ditarik Kembali,scraping_kelas
4,zeolite.id,zeolite id,zeolite id,(TM) Dianggap Ditarik Kembali,scraping_kelas


## 5. Terapkan preprocessing ke gold cases

In [7]:
gold = gold_raw.copy()
for kol in ["merek_a", "merek_b"]:
    gold[f"{kol}_base"] = gold[kol].apply(base_normalize)
    gold[f"{kol}_phon"] = gold[f"{kol}_base"].apply(lambda b: phonetic_normalize(b, AKTIFKAN_FONETIK_ID))
print("Gold diproses:", gold.shape)
gold[["merek_a","merek_a_base","merek_a_phon","merek_b","merek_b_base","merek_b_phon"]].head(8)

Gold diproses: (95, 18)


,merek_a,merek_a_base,merek_a_phon,merek_b,merek_b_base,merek_b_phon
0,KOJIE-SAN,kojie san,kojie san,KOJIE-SAN DREAM WHITE,kojie san dream white,kojie san dream white
1,I AM GEPREK BENSU SEDEP BENEEERRR,i am geprek bensu sedep beneeerrr,i am geprek bensu sedep beneeerrr,I AM GEPREK BENSU,i am geprek bensu,i am geprek bensu
2,JOLLIBEE,jollibee,jollibee,JOLIBI,jolibi,jolibi
3,J. CASANOVA,j casanova,j casanova,CASANOVA,casanova,casanova
4,PS GLOW,ps glow,ps glow,MS GLOW,ms glow,ms glow
5,POLOBYRALPHLAUREN,polobyralphlauren,polobyralphlauren,POLO BY RALPH LAUREN,polo by ralph lauren,polo by ralph lauren
6,HUGO BOSS,hugo boss,hugo boss,HUGO SELECT LINE,hugo select line,hugo select line
7,TPC/TPC PNEUMATICS,tpc tpc pneumatics,tpc tpc pneumatics,TPC PNEUMATICS,tpc pneumatics,tpc pneumatics


## 6. Simpan hasil

In [8]:
suffix = "fonID" if AKTIFKAN_FONETIK_ID else "noFonID"
hay_valid.to_csv(f"haystack_preprocessed_{suffix}.csv", index=False, encoding="utf-8-sig")
hay_didaftar.to_csv(f"haystack_didaftar_{suffix}.csv", index=False, encoding="utf-8-sig")
gold.to_csv(f"gold_cases_preprocessed_{suffix}.csv", index=False, encoding="utf-8-sig")
print("Tersimpan:")
print(f"  haystack_preprocessed_{suffix}.csv  ({len(hay_valid)}) — corpus FastText")
print(f"  haystack_didaftar_{suffix}.csv      ({len(hay_didaftar)}) — basis pembanding retrieval")
print(f"  gold_cases_preprocessed_{suffix}.csv({len(gold)}) — gold cases + kolom base/phon")

Tersimpan:
  haystack_preprocessed_noFonID.csv  (13514) — corpus FastText
  haystack_didaftar_noFonID.csv      (7650) — basis pembanding retrieval
  gold_cases_preprocessed_noFonID.csv(95) — gold cases + kolom base/phon


## 7. Simpan modul .py (agar Langkah C impor fungsi yang sama)

In [9]:
modul = '''import re, unicodedata
LEET = {"0":"o","1":"i","3":"e","4":"a","5":"s","7":"t"}
FIGURATIF = r"(lukisan|logo|gambar|device)"
PHONETIC_RULES = [("sy","sh"), ("ny","n"), ("kh","h")]
def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(c))
def base_normalize(nama):
    s = strip_accents(nama).lower()
    s = re.sub(r"\\+\\s*" + FIGURATIF + r"\\b", " ", s)
    s = re.sub(r"\\b" + FIGURATIF + r"\\b", " ", s)
    s = re.sub(r"(?<=[a-z])[013457](?=[a-z])", lambda m: LEET.get(m.group(0), m.group(0)), s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\\s+", " ", s).strip()
def phonetic_normalize(base_str, aktif=True):
    if not aktif: return base_str
    x = base_str
    for a, b in PHONETIC_RULES: x = x.replace(a, b)
    x = re.sub(r"([aeiou])\\1+", r"\\1", x)
    return re.sub(r"\\s+", " ", x).strip()
def preprocess_nama(nama, aktif_fonetik=True):
    b = base_normalize(nama)
    return {"nama_base": b, "nama_phon": phonetic_normalize(b, aktif_fonetik)}
'''
with open("preprocessing_merek.py","w",encoding="utf-8") as f: f.write(modul)
import importlib, preprocessing_merek; importlib.reload(preprocessing_merek)
assert preprocessing_merek.base_normalize("Cafe + LOGO") == "cafe"
print("preprocessing_merek.py tersimpan & terverifikasi.")

preprocessing_merek.py tersimpan & terverifikasi.


## 8. Unduh
**Lanjut ke berikutnya — ekstraksi fitur** (Jaro-Winkler & FastText pada `nama_base`, Double Metaphone pada `nama_phon`).


In [10]:
from google.colab import files
suffix = "fonID" if AKTIFKAN_FONETIK_ID else "noFonID"
for f in [f"haystack_preprocessed_{suffix}.csv", f"haystack_didaftar_{suffix}.csv",
          f"gold_cases_preprocessed_{suffix}.csv", "preprocessing_merek.py"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>